In [24]:
from icw.auto_watermark import AutoWatermark
from datasets import load_dataset
from tqdm import tqdm
from evaluation.evaluation import Evaluation
import pandas as pd

In [ ]:
import os

EXTRA_INSTRUCTION = "Please answer the question in about 300 words."

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')


unicode_config = {
    'icw': 'UNICODE',
    'model': 'o3-mini',
    'model_type': 'r.high',   # 'r.high' for high reasoning mode; 'n.n' for non-reasoning
    'api_key': OPENAI_API_KEY,
    'wm_instruction': None, # if None, using the default instruction
    'extra_instruction': EXTRA_INSTRUCTION,
}

lexical_config = {
    'icw': 'LEXICAL',
    'model': 'o3-mini',
    'model_type': 'r.high',   # 'r.high' for reasoning; 'n.n' for non-reasoning
    'api_key': OPENAI_API_KEY,
    'seed': 123,
    'gamma': 0.20,
    'r_high_freq': 300,  
    'r_low_freq': 13000,
    'wm_instruction': None, # if None, using the default instruction
    'extra_instruction': EXTRA_INSTRUCTION,
}

initials_config = {
    'icw': 'INITIALS',
    'model': 'o3-mini',
    'model_type': 'r.high',   # 'r.high' for reasoning; 'n.n' for non-reasoning
    'api_key': OPENAI_API_KEY,
    'wm_instruction': None, # if None, using the default instruction
    'extra_instruction': EXTRA_INSTRUCTION,
}

acrostics_config = {
    'icw': 'ACROSTICS',
    'model': 'o3-mini',
    'model_type': 'r.high',   # 'r.high' for reasoning; 'n.n' for non-reasoning
    'seed': 123,
    'str_len': 20,
    'api_key': OPENAI_API_KEY,
    'wm_instruction': None, # if None, using the default instruction
    'extra_instruction': EXTRA_INSTRUCTION,
}

eva_config = {
    'model': 'gpt-4o-mini',
    'api_key': OPENAI_API_KEY,
}

In [4]:
unicodeWatermark = AutoWatermark.load(unicode_config)

In [ ]:
lexicalWatermark = AutoWatermark.load(lexical_config)

In [ ]:
initialsWatermark = AutoWatermark.load(initials_config)

In [26]:
acrosticsWatermark = AutoWatermark.load(acrostics_config)

In [6]:
evaluation = Evaluation(eva_config)

### Direct Text Stamp (DTS) Setting

In [ ]:
# icw = unicodeWatermark
# icw = lexicalWatermark
icw = initialsWatermark
# icw = acrosticsWatermark


ds = pd.read_csv('dataset/eli5.csv')
row =[]
for i in range(len(ds)):
    prompt = ds.iloc[i]['question']
    reference_answer = ds.iloc[i]['reference_answer']

    unwm_score = icw.detect_watermark(reference_answer)

    wm_text = icw.generate_watermarked_text(prompt)
    wm_score = icw.detect_watermark(wm_text)

    # p_wm_text = evaluation.paraphrase(wm_text)
    # p_wm_score = icw.detect_watermark(p_wm_text)

    # d_wm_text = evaluation.random_word_deletion(wm_text)
    # d_wm_score = icw.detect_watermark(d_wm_text)

    # r_wm_text = evaluation.random_word_replacement(wm_text)
    # r_wm_score = icw.detect_watermark(r_wm_text)

    row.append({
        'question': prompt,
        'reference_answer': reference_answer,
        'wm_response': wm_text,
        # 'd_response': d_wm_text,
        # 'r_response': r_wm_text,
        # 'p_response': p_wm_text,
        'unwm_score': unwm_score,
        'wm_score': wm_score,
        # 'd_score': d_wm_score,
        # 'r_score': r_wm_score,
        # 'p_score': p_wm_score
    })
    df = pd.DataFrame(row)
    df.to_csv(f'outputs/dts/{type(icw).__name__}_dts.csv', index=False)

